# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

# As an alternative, if you'd like to use Ollama instead of OpenAI
# Check that Ollama is running for you locally (see week1/day2 exercise) then uncomment these next 2 lines
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


OpenAI API Key exists and begins sk-proj-


In [3]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [4]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky.. we're giving it the power to run code on our machine?

Well, kinda.

In [5]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"


In [6]:
get_ticket_price("London")

Tool called for city London


'The price of a ticket to London is $799'

In [7]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [8]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}]

In [9]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [10]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [11]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [12]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


Tool called for city Paris


## Let's make a couple of improvements

Handling multiple tool calls in 1 response

Handling multiple tool calls 1 after another

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [13]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [14]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


Tool called for city Paris


In [15]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [16]:
import sqlite3


In [23]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [24]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [25]:
get_ticket_price("London")

DATABASE TOOL CALLED: Getting price for London


'No price data available for this city'

In [27]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [28]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [29]:
get_ticket_price("Tokyo")

DATABASE TOOL CALLED: Getting price for Tokyo


'Ticket price to Tokyo is $1420.0'

In [30]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for Paris


## Exercise

Add a tool to set the price of a ticket!

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Hopefully this hardly needs to be stated! You now have the ability to give actions to your LLMs. This Airline Assistant can now do more than answer questions - it could interact with booking APIs to make bookings!</span>
        </td>
    </tr>
</table>

In [31]:
import os
import json
import sqlite3
import secrets
import datetime as dt
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)
client = OpenAI()
MODEL = "gpt-4.1-mini"
DB = "bookings.db"


def init_db():
    with sqlite3.connect(DB) as conn:
        c = conn.cursor()
        c.execute(
            """CREATE TABLE IF NOT EXISTS flights (
            flight_id TEXT PRIMARY KEY, origin TEXT, destination TEXT,
            departure_date TEXT, departure_time TEXT,
            price REAL, seats_available INTEGER)"""
        )
        c.execute(
            """CREATE TABLE IF NOT EXISTS bookings (
            booking_ref TEXT PRIMARY KEY, flight_id TEXT,
            passenger_name TEXT, passenger_email TEXT,
            seats INTEGER, status TEXT, created_at TEXT)"""
        )
        c.execute("SELECT COUNT(*) FROM flights")
        if c.fetchone()[0] == 0:
            c.executemany(
                "INSERT INTO flights VALUES (?,?,?,?,?,?,?)",
                [
                    ("FA101", "London", "Paris", "2026-06-12", "09:00", 799, 12),
                    ("FA102", "London", "Paris", "2026-06-12", "18:30", 749, 20),
                    ("FA201", "London", "Tokyo", "2026-06-15", "11:00", 1420, 6),
                    ("FA301", "London", "Berlin", "2026-06-13", "07:45", 499, 30),
                    ("FA401", "London", "Sydney", "2026-06-20", "21:10", 2999, 4),
                ],
            )
        conn.commit()


init_db()


def search_flights(origin, destination, departure_date):
    with sqlite3.connect(DB) as conn:
        rows = conn.execute(
            """SELECT flight_id, departure_time, price, seats_available
               FROM flights
               WHERE LOWER(origin)=? AND LOWER(destination)=? AND departure_date=?""",
            (origin.lower(), destination.lower(), departure_date),
        ).fetchall()
    return [{"flight_id": r[0], "time": r[1], "price": r[2], "seats_available": r[3]} for r in rows]


def book_flight(flight_id, passenger_name, passenger_email, seats=1):
    with sqlite3.connect(DB) as conn:
        c = conn.cursor()
        c.execute("SELECT seats_available, price FROM flights WHERE flight_id=?", (flight_id,))
        row = c.fetchone()
        if not row:
            return {"error": f"Unknown flight {flight_id}"}
        if row[0] < seats:
            return {"error": "Not enough seats available"}
        ref = secrets.token_hex(3).upper()
        c.execute(
            "UPDATE flights SET seats_available=seats_available-? WHERE flight_id=?",
            (seats, flight_id),
        )
        c.execute(
            "INSERT INTO bookings VALUES (?,?,?,?,?,?,?)",
            (
                ref,
                flight_id,
                passenger_name,
                passenger_email,
                seats,
                "confirmed",
                dt.datetime.utcnow().isoformat(),
            ),
        )
        conn.commit()
    return {
        "booking_ref": ref,
        "status": "confirmed",
        "flight_id": flight_id,
        "passenger": passenger_name,
        "seats": seats,
        "total_price": row[1] * seats,
    }


tools = [
    {
        "type": "function",
        "function": {
            "name": "search_flights",
            "description": "Search available flights by origin, destination and date.",
            "parameters": {
                "type": "object",
                "properties": {
                    "origin": {"type": "string", "description": "Origin city, e.g. London"},
                    "destination": {"type": "string", "description": "Destination city, e.g. Paris"},
                    "departure_date": {"type": "string", "description": "Departure date in YYYY-MM-DD"},
                },
                "required": ["origin", "destination", "departure_date"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "book_flight",
            "description": "Book a specific flight for a passenger. Only call after the user has confirmed the details.",
            "parameters": {
                "type": "object",
                "properties": {
                    "flight_id": {"type": "string"},
                    "passenger_name": {"type": "string"},
                    "passenger_email": {"type": "string"},
                    "seats": {"type": "integer", "minimum": 1, "default": 1},
                },
                "required": ["flight_id", "passenger_name", "passenger_email"],
                "additionalProperties": False,
            },
        },
    },
]

TOOL_FUNCTIONS = {
    "search_flights": lambda a: search_flights(a["origin"], a["destination"], a["departure_date"]),
    "book_flight": lambda a: book_flight(
        a["flight_id"], a["passenger_name"], a["passenger_email"], a.get("seats", 1)
    ),
}

system_message = (
    "You are a helpful assistant for an airline called FlightAI. "
    "You can search flights and create bookings using the provided tools. "
    "Before calling book_flight, always read back the flight, passenger name, "
    "and email in one short message and wait for the user to confirm. "
    "Give short, courteous answers. "
    "If you dont know the answer, say so."
)


def handle_tool_calls(message):
    out = []
    for tc in message.tool_calls:
        args = json.loads(tc.function.arguments)
        try:
            result = TOOL_FUNCTIONS[tc.function.name](args)
        except Exception as e:
            result = {"error": str(e)}
        out.append(
            {
                "role": "tool",
                "content": json.dumps(result, default=str),
                "tool_call_id": tc.id,
            }
        )
    return out


def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        msg = response.choices[0].message
        messages.append(msg)
        messages.extend(handle_tool_calls(msg))
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    return response.choices[0].message.content


gr.ChatInterface(fn=chat, type="messages").launch()


* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


/var/folders/2h/pr69y8wx3v3gsn_gvrwvxhr40000gn/T/ipykernel_78435/3300790042.py:83: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  dt.datetime.utcnow().isoformat(),
